# ST-03 — Siretisation Phase 2 (top 5 probabiliste)

Sur les EG du périmètre ST-02, recherche des **5 meilleurs établissements SIRENE candidats** par EG. Blocking sur le code commune.

**Bonus +15** sur le `score_global_ajuste` si le SIRET candidat = SIRET FINESS de l'EG. Désactivé si le SIRET FINESS est déjà validé en P1 (cas des EG jumeaux).

Statuts : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_CANDIDAT

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from tqdm.auto import tqdm

from src.siretisation import scorer_paire_eg_etab
from src.matching     import classifier_resultat
from src.excel_export import export_topn_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    ST_PERIMETRE, SIRENE_ETAB_FILTRE, ST_PHASE1, ST_PHASE2, RESULTS_ST_DIR,
)
RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)

# Récupération des SIRET validés en P1 (pour désactivation bonus)
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}
sheets_p1 = pd.read_excel(ST_PHASE1, sheet_name=None, dtype=str)
sirets_valides_p1 = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES and 'nmsiret_stru' in sdf.columns:
        sirets_valides_p1.update(
            sdf['nmsiret_stru'].dropna().astype(str)
            .str.replace(r'\s', '', regex=True).str.strip()
        )
sirets_valides_p1.discard('')
print(f'SIRET validés en P1 (pour désactivation bonus) : {len(sirets_valides_p1):,}')

SIRET validés en P1 (pour désactivation bonus) : 56,992


## 1. Chargement

In [2]:
df_perimetre = pd.read_parquet(ST_PERIMETRE)
df_etab      = pd.read_parquet(SIRENE_ETAB_FILTRE)
df_etab['siret'] = df_etab['siret'].astype(str)

print(f'EG à traiter Phase 2     : {len(df_perimetre):,}')
print(f'Etab SIRENE candidats    : {len(df_etab):,}  (base filtrée)')

EG à traiter Phase 2     : 44,886
Etab SIRENE candidats    : 241,627  (base filtrée)


## 2. Indexation des Etab par commune

In [3]:
etab_par_commune = df_etab.groupby('code_commune_norm_etab', sort=False)
print(f'Communes avec au moins un Etab : {etab_par_commune.ngroups:,}')

Communes avec au moins un Etab : 13,065


## 3. Recherche des top 5 candidats

In [4]:
BONUS_SIRET_COHERENT = 15.0

lignes_top5 = []
lignes_orphelins = []

for _, row_eg in tqdm(df_perimetre.iterrows(), total=len(df_perimetre),
                       desc='Phase 2 siretisation'):
    commune = row_eg['cdcommune_norm_eg']
    siret_eg = str(row_eg.get('nmsiret_stru', '') or '').strip().replace(' ', '')
    bonus_applicable = (siret_eg != '') and (siret_eg not in sirets_valides_p1)

    if not commune or commune not in etab_par_commune.groups:
        lignes_orphelins.append({
            **row_eg.to_dict(),
            'siret_ref': None, 'siren_ref': None, 'nom_etab_retenu': None,
            'score_nom': None, 'score_adresse': None, 'score_global': None,
            'siret_coherent': False, 'bonus_applique': False,
            'score_global_ajuste': None,
            'rang': 1, 'statut_candidat': 'SANS_CANDIDAT',
        })
        continue

    candidates = etab_par_commune.get_group(commune)

    scores = []
    for _, row_etab in candidates.iterrows():
        s = scorer_paire_eg_etab(row_eg, row_etab)
        siret_etab = str(row_etab['siret']).strip()
        coherent = bool(siret_eg) and (siret_eg == siret_etab)
        bonus = BONUS_SIRET_COHERENT if (coherent and bonus_applicable) else 0.0
        score_ajuste = min(s['score_global'] + bonus, 100.0)

        scores.append({
            'siret_ref':                     row_etab['siret'],
            'siren_ref':                     row_etab.get('siren'),
            'denominationUniteLegale':       row_etab.get('denominationUniteLegale'),
            'enseigne1Etablissement':        row_etab.get('enseigne1Etablissement'),
            'enseigne2Etablissement':        row_etab.get('enseigne2Etablissement'),
            'enseigne3Etablissement':        row_etab.get('enseigne3Etablissement'),
            'denominationUsuelleEtablissement': row_etab.get('denominationUsuelleEtablissement'),
            'adresse_complete_etab':         row_etab.get('adresse_complete_etab'),
            'codeCommuneEtablissement':      row_etab.get('codeCommuneEtablissement'),
            'categorieJuridiqueUniteLegale': row_etab.get('categorieJuridiqueUniteLegale'),
            'activitePrincipaleUniteLegale': row_etab.get('activitePrincipaleUniteLegale'),
            'dateCreationEtablissement':     row_etab.get('dateCreationEtablissement'),
            **s,
            'siret_coherent':       coherent,
            'bonus_applique':       bool(bonus > 0),
            'score_global_ajuste':  round(score_ajuste, 2),
        })

    scores.sort(key=lambda x: x['score_global_ajuste'], reverse=True)
    for rang, sc in enumerate(scores[:5], start=1):
        statut = classifier_resultat(
            sc['score_global_ajuste'], sc['score_nom'], sc['score_adresse']
        )
        lignes_top5.append({
            **row_eg.to_dict(), **sc,
            'rang': rang, 'statut_candidat': statut,
        })

df_top5 = pd.DataFrame(lignes_top5 + lignes_orphelins)

print(f'\nLignes top 5 : {len(df_top5):,}')
print(df_top5[df_top5['rang'] == 1]['statut_candidat'].value_counts())

Phase 2 siretisation:   0%|          | 0/44886 [00:00<?, ?it/s]


Lignes top 5 : 218,794
statut_candidat
VALIDE           26960
DOUTEUX          10714
VALIDE_FORT       4764
REJETE            2218
SANS_CANDIDAT      230
Name: count, dtype: int64


## 4. Aperçu

In [5]:
afficher_tableau(
    df_top5[['idstructure_stru', 'nmfinessej_stru', 'raisonsociale_stru',
             'siret_ref', 'denominationUniteLegale', 'nom_etab_retenu',
             'score_nom', 'score_adresse', 'score_global', 'score_global_ajuste',
             'statut_candidat', 'rang']],
    'Aperçu top 5 Phase 2', max_lignes=10,
)

idstructure_stru,nmfinessej_stru,raisonsociale_stru,siret_ref,denominationUniteLegale,nom_etab_retenu,score_nom,score_adresse,score_global,score_global_ajuste,statut_candidat,rang
1931888,240011262,PHARMACIE REYDY,43352756100016,None,,0.000000,100.000000,60.000000,75.000000,VALIDE,1
1931888,240011262,PHARMACIE REYDY,47919175100011,ASSO. COMMERC REPRODUCTION CAPRINS,COMMERC REPRODUCTION CAPRINS,26.050000,100.000000,70.420000,70.420000,VALIDE,2
1931888,240011262,PHARMACIE REYDY,83384101800017,SCI PLACE HOCHE,PLACE HOCHE,36.620000,80.230000,62.780000,62.780000,VALIDE,3
1931888,240011262,PHARMACIE REYDY,35237715400021,LA POMMERAIE SA,POMMERAIE,41.670000,69.930000,58.630000,58.630000,VALIDE,4
1931888,240011262,PHARMACIE REYDY,79148134400285,A2MICILE REGION CENTRE,A2MICILE REGION CENTRE,31.570000,72.390000,56.060000,56.060000,VALIDE,5
1931893,240011346,PHARMACIE DAUMARES,44236514400017,SYNDICAT COPROP SAINT MARTIN,SYNDICAT COPROP SAINT MARTIN,12.520000,87.150000,57.300000,57.300000,DOUTEUX,1
1931893,240011346,PHARMACIE DAUMARES,90056938500017,PHARMACIE CHAPARD,PHARMACIE CHAPARD,54.740000,57.150000,56.190000,56.190000,VALIDE,2
1931893,240011346,PHARMACIE DAUMARES,44454081900018,DA COSTA,DA COSTA,28.310000,67.110000,51.590000,51.590000,VALIDE,3
1931893,240011346,PHARMACIE DAUMARES,52805300200010,SARL DA COSTA,DA COSTA,28.310000,67.110000,51.590000,51.590000,VALIDE,4
1931893,240011346,PHARMACIE DAUMARES,82773193600023,PATRIMONIA,PATRIMONIA,42.100000,57.150000,51.130000,51.130000,VALIDE,5


## 5. Export Excel

In [6]:
COLS_EXPORT = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru',
    'categetab_stru', 'nmsiret_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouvertstruct_stru',
    'cdcommune_stru', 'adresse_complete_eg',
    'siret_ref', 'siren_ref', 'siret_coherent', 'bonus_applique',
    'denominationUniteLegale', 'enseigne1Etablissement', 'enseigne2Etablissement',
    'enseigne3Etablissement', 'denominationUsuelleEtablissement', 'nom_etab_retenu',
    'adresse_complete_etab', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'dateCreationEtablissement',
    'score_nom', 'score_adresse', 'score_global', 'score_global_ajuste',
    'statut_candidat', 'rang',
]

df_top5['rang'] = df_top5['rang'].astype(int)
compteurs = export_topn_excel(df_top5, ST_PHASE2, COLS_EXPORT, sheet_name='Top5')

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Phase 2 siretisation (rang 1)')
print(f'\nFichier : {ST_PHASE2}')

Statut,Nb,% du total
Valide_fort,"4,764",10.6%
Valide,"26,960",60.1%
Douteux,"10,714",23.9%
Rejeté,"2,218",4.9%
Sans_candidat,230,0.5%
TOTAL,"44,886",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/siretisation/siretisation_phase2_top5.xlsx
